# MEdAI: train the baseline on Colab's free GPU

Run the cells **top to bottom**, one at a time (Shift + Enter).

**Before you start (one time only)**
1. `Runtime → Change runtime type → T4 GPU → Save`.
2. Create a GitHub token so Colab can read the private repo:
   GitHub → Settings → Developer settings → Personal access tokens → **Fine-grained tokens** →
   Generate new token → Repository access: *Only select repositories* → `medai` →
   Permissions: **Contents: Read and write** → Generate. Copy it.
3. In Colab, click the **key icon (Secrets)** on the left → *Add new secret* →
   Name `GITHUB_TOKEN`, Value = the token → switch **Notebook access** on.

**If Colab disconnects:** reconnect, run every cell again from the top.
Training continues from the last finished epoch, because checkpoints are saved to your Google Drive.

## 1 · Check the GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> T4 GPU, then run again."
print("GPU:", torch.cuda.get_device_name(0))

## 2 · Connect Google Drive (checkpoints are saved here so a disconnect loses nothing)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3 · Get the code

In [ ]:
import os
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
REPO = '/content/medai'
if not os.path.exists(REPO):
    !git clone -q https://{token}@github.com/praneethnarne/medai {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}
!git log --oneline -1

## 4 · Install dependencies (Colab already has PyTorch; this adds the rest)

In [ ]:
!pip install -q -r backend/requirements.txt

## 5 · Download the datasets from the ISIC archive

Takes several minutes. Safe to re-run: images already downloaded are skipped.
The committed `splits.json` files are used as-is, so the split is identical to the one in the repo.

In [ ]:
!python backend/src/data/download.py pad_ufes_20 --workers 16
!python backend/src/data/download.py ham10000 --workers 16
!python backend/src/data/verify.py ham10000 pad_ufes_20

## 6 · Point the run folder at Google Drive

In [ ]:
import pathlib, shutil
RUN = 'ham_effb0_colab'   # change only if you want a separate run
drive_dir = pathlib.Path('/content/drive/MyDrive/medai_runs') / RUN
drive_dir.mkdir(parents=True, exist_ok=True)
local = pathlib.Path('results/runs') / RUN
local.parent.mkdir(parents=True, exist_ok=True)
if local.is_symlink():
    pass
elif local.exists():
    shutil.rmtree(local)
if not local.exists():
    local.symlink_to(drive_dir)
print(local, '->', drive_dir)
print('already in Drive:', sorted(p.name for p in drive_dir.iterdir()))

## 7 · Train

Stage 1 trains only the new output layer (3 epochs); stage 2 unfreezes the whole network (up to 15 epochs,
stops early if validation macro F1 stops improving for 5 epochs).

`--resume` is harmless on the first run (it just starts fresh) and continues from the last epoch after a disconnect.
Keep this browser tab open while it runs.

In [ ]:
!python -u backend/src/train/train.py \
    --collection ham10000 --target lesion_type --arch efficientnet_b0 \
    --epochs-head 3 --epochs-full 15 --workers 2 \
    --run-name {RUN} --resume

## 8 · Evaluate

First on HAM10000's own test split (in-domain), then on real phone photos from PAD-UFES-20 (cross-domain).
The gap between the two is the project's headline result. Report it whatever it says.

In [ ]:
!python backend/src/eval/evaluate.py --run {RUN} --collection ham10000 --split test

In [ ]:
!python backend/src/eval/evaluate.py --run {RUN} --collection pad_ufes_20 --split test --collapse-to-triage

## 9 · Save the results to GitHub

Pushes the run's JSON files (config, history, metrics) and the evaluation results to a branch called `colab-results`.
Safe to run more than once. Run step 6 again afterwards before launching the demo.
The model weights (`best.pt`, `last.pt`) are too large for git and stay in Google Drive under `medai_runs/`.

In [ ]:
out = pathlib.Path('/content/results_push') / RUN
if out.exists():
    shutil.rmtree(out)
out.mkdir(parents=True)
for f in drive_dir.glob('*.json'):
    shutil.copy(f, out / f.name)
if local.is_symlink():              # replace the Drive link with real files for git
    local.unlink()
    shutil.copytree(out, local)     # (already a plain copy if this cell ran before)
!git checkout -q -B colab-results
!git add results/runs/{RUN} results/eval
!git -c user.name="praneethnarne" -c user.email="praneethnarne@users.noreply.github.com" commit -q -m "Add Colab baseline run {RUN}"
!git push -q -f https://{token}@github.com/praneethnarne/medai colab-results
print('Pushed. Open the repo on GitHub and switch to the colab-results branch.')

## 10 · Launch the demo

Installs Gradio and starts the app on the best checkpoint. Look for a line
`Running on public URL: https://....gradio.live` and open it -- that link works
on any device (laptop, phone, projector) for up to 72 hours while this cell runs.
Stop it with the stop button on the cell.

If step 9 was run, the run folder is a plain copy; re-run step 6 first so the
demo reads `best.pt` from Drive.

In [ ]:
!pip install -q gradio
!python backend/src/demo/app.py --run {RUN} --share

# Part 2 · Make it reliable on phone photos

Steps 11-14 fine-tune the HAM10000 model on real phone photos (PAD-UFES-20), test whether
simulated phone-camera effects help, then calibrate the confidence scores. Needs steps 1-4
and 6 run in this session. If the runtime is fresh, also run step 5 (only `pad_ufes_20` is needed).

## 11 · Fine-tune on phone photos -- two runs

Both start from the HAM10000 model. Run A trains on PAD-UFES-20 as it is; run B adds simulated
phone-camera effects (colour cast, blur, low resolution, JPEG compression). Comparing them tests
whether those effects actually help. Progress is saved to Drive, so `--resume` continues after a disconnect.

In [ ]:
def link_run(name):
    d = pathlib.Path('/content/drive/MyDrive/medai_runs') / name
    d.mkdir(parents=True, exist_ok=True)
    l = pathlib.Path('results/runs') / name
    if l.exists() and not l.is_symlink():
        shutil.rmtree(l)
    if not l.exists():
        l.symlink_to(d)

for name in [RUN, 'pad_ft_effb0', 'pad_ft_effb0_phoneaug']:
    link_run(name)
assert (pathlib.Path('results/runs') / RUN / 'best.pt').exists(), 'HAM model missing: run step 6'

!python -u backend/src/train/train.py --collection pad_ufes_20 --target lesion_type \
    --arch efficientnet_b0 --epochs-head 2 --epochs-full 15 --workers 2 \
    --init-from results/runs/{RUN}/best.pt --run-name pad_ft_effb0 --resume

In [ ]:
!python -u backend/src/train/train.py --collection pad_ufes_20 --target lesion_type \
    --arch efficientnet_b0 --epochs-head 2 --epochs-full 15 --workers 2 \
    --init-from results/runs/{RUN}/best.pt --run-name pad_ft_effb0_phoneaug --phone-aug --resume

## 12 · Pick the better run on VALIDATION, then test both

The choice uses validation data only, so the test numbers stay honest. Both runs are still
tested and reported, because "did phone effects help?" is a result either way.

In [ ]:
import json
scores = {r: json.load(open(f'results/runs/{r}/best_val_report.json'))['macro_f1']
          for r in ['pad_ft_effb0', 'pad_ft_effb0_phoneaug']}
for r, f in scores.items():
    print(f'{r:28s} validation macro F1 = {f:.4f}')
BEST = max(scores, key=scores.get)
print('\nchosen on validation:', BEST)

In [ ]:
for r in ['pad_ft_effb0', 'pad_ft_effb0_phoneaug']:
    !python backend/src/eval/evaluate.py --run {r} --collection pad_ufes_20 --split test --collapse-to-triage

## 13 · Calibrate the chosen run and set the "not confident" threshold

Fits a temperature on validation data so the scores mean what they say, then picks the lowest
threshold at which at least 90% of malignant validation cases are either flagged malignant or
declined (sent to a doctor). Reports the result on the test set.

In [ ]:
!python backend/src/eval/calibrate.py --run {BEST} --collection pad_ufes_20 --target-recall 0.90

## 14 · Back up the results, then launch the demo with the new model

In [ ]:
!mkdir -p /content/drive/MyDrive/medai_runs/eval && cp results/eval/*.json /content/drive/MyDrive/medai_runs/eval/
!pip install -q gradio
!python backend/src/demo/app.py --run {BEST} --share

## 15 · "I don't recognise this image" check

Fits a familiarity check on the chosen run's training images: an image whose features are far
from every lesion type the model learned is answered with "I don't recognise this image"
instead of a guess. The threshold lets 95% of real validation lesions through. How many scars,
rashes or non-skin photos it catches is not measured -- there is no labelled set of those.

In [ ]:
!python backend/src/eval/ood.py --run pad_ft_effb0_phoneaug --fit pad_ufes_20 --check pad_ufes_20 ham10000
!python backend/src/demo/app.py --run pad_ft_effb0_phoneaug --share

# Part 3 · What kind of thing is on the skin? (ADR 0007)

Steps 16-20 train the Level 1 model: spot or growth · rash · infection · bite or sting ·
wound, burn, bruise or scar. It learns from PAD-UFES-20 plus Google's SCIN dataset (phone photos
of common skin conditions, labelled by dermatologists). Needs steps 1-4 and 6 in this session,
and PAD-UFES-20 images (step 5) if the runtime is fresh.

## 16 · Download SCIN (~5,900 photos, shrunk to 512 px, ~0.2 GB)

In [ ]:
!python backend/src/data/scin.py images --workers 16
!python backend/src/data/scin.py audit

## 17 · Train the Level 1 model

Starts from ImageNet. No phone-camera effects here: SCIN is already phone photos, and the effects
would make each epoch about three times slower on Colab's two CPU cores. Saved to Drive as it goes.

In [ ]:
import pathlib, shutil
def link_run(name):
    d = pathlib.Path('/content/drive/MyDrive/medai_runs') / name
    d.mkdir(parents=True, exist_ok=True)
    l = pathlib.Path('results/runs') / name
    if l.exists() and not l.is_symlink():
        shutil.rmtree(l)
    if not l.exists():
        l.symlink_to(d)
link_run('groups_effb0')
!python -u backend/src/train/train.py --target group --arch efficientnet_b0 \
    --epochs-head 2 --epochs-full 15 --workers 2 --run-name groups_effb0 --resume

## 18 · Test it

Prints the full test table, then the same table on SCIN images only. The second one is a shortcut
check: almost every spot/growth training image comes from PAD-UFES-20, so if spot/growth recall
collapses on SCIN's own spots, the model learned "looks like a PAD photo" rather than "is a spot".

In [ ]:
!python backend/src/eval/evaluate.py --run groups_effb0 --collection groups --split test

## 19 · Calibrate, and fit the unfamiliar-image check for the Level 1 model

In [ ]:
!python backend/src/eval/calibrate.py --run groups_effb0 --target-accuracy 0.90
!python backend/src/eval/ood.py --run groups_effb0 --fit groups --check groups ham10000

## 20 · Demo with both levels

In [ ]:
!cp results/eval/*.json /content/drive/MyDrive/medai_runs/eval/ 2>/dev/null; true
!python backend/src/demo/app.py --groups-run groups_effb0 --run pad_ft_effb0_phoneaug --share

# Part 4 · One model for both kinds of photo (joint training)

The deployed model was fine-tuned on phone photos and now over-flags dermatoscope images
(malignant precision 0.293 on the HAM10000 test set). Steps 21-24 train one model on
HAM10000 **and** PAD-UFES-20 together. Each image keeps its own patient- or lesion-level split.

**Run steps 1-5 first** (GPU, Drive, code, install, datasets). The run is resumable: if
`MyDrive/medai_runs/joint_effb0_phoneaug/last.pt` exists (e.g. uploaded from the Mac), it
continues from there instead of starting over.

## 21 · Switch to the joint-training code

In [ ]:
!git fetch -q origin joint-training && git checkout -q joint-training && git pull -q
!git log --oneline -1

## 22 · Point the run folders at Drive, then train (resumes automatically)

In [ ]:
import pathlib, shutil
def link_run(name):
    d = pathlib.Path('/content/drive/MyDrive/medai_runs') / name
    d.mkdir(parents=True, exist_ok=True)
    l = pathlib.Path('results/runs') / name
    if l.exists() and not l.is_symlink():
        shutil.rmtree(l)
    if not l.exists():
        l.symlink_to(d)

for name in ['ham_effb0_colab', 'joint_effb0_phoneaug']:
    link_run(name)
assert pathlib.Path('results/runs/ham_effb0_colab/best.pt').exists(), 'HAM model missing in Drive'
print('resuming' if pathlib.Path('results/runs/joint_effb0_phoneaug/last.pt').exists() else 'starting fresh')

!python -u backend/src/train/train.py --collection ham10000+pad_ufes_20 --target lesion_type \
    --arch efficientnet_b0 --epochs-head 2 --epochs-full 15 --workers 2 \
    --init-from results/runs/ham_effb0_colab/best.pt --phone-aug \
    --run-name joint_effb0_phoneaug --resume

## 23 · Calibrate on phone photos, test on both domains, fit the unfamiliar-image check

The question this answers: does one model fix the dermatoscope over-flagging **without**
losing phone-photo accuracy? Compare with `pad_ft_effb0_phoneaug` in `results/eval/`.

In [ ]:
!python backend/src/eval/calibrate.py --run joint_effb0_phoneaug --collection pad_ufes_20 --target-recall 0.90
!python backend/src/eval/evaluate.py --run joint_effb0_phoneaug --collection pad_ufes_20 --split test --collapse-to-triage
!python backend/src/eval/evaluate.py --run joint_effb0_phoneaug --collection ham10000 --split test --collapse-to-triage
!python backend/src/eval/ood.py --run joint_effb0_phoneaug --fit pad_ufes_20 ham10000 --check pad_ufes_20 ham10000

## 24 · Back up the evaluation files to Drive

In [ ]:
!mkdir -p /content/drive/MyDrive/medai_runs/eval && cp results/eval/*.json /content/drive/MyDrive/medai_runs/eval/
!ls -la /content/drive/MyDrive/medai_runs/joint_effb0_phoneaug